# CUTS+ on Causal Chambers wind-tunnel measurements

This notebook is the `causalChamber` copy of the `cuts-plus` experiment.
It reads the same frozen, hash-verified bundles as every other method in this
directory and writes one atomic checkpoint per completed dataset fit.

Source: **jarrycyx/UNN**. Comparison methods use a pinned installed
distribution when upstream is packageable; repository-only methods call a
verified checkout. Their estimator code is neither copied nor reimplemented
here. Native arrays are saved before common graph-orientation conversion and visualization. This empirical case study compares the inferred directed graphs qualitatively; recovery metrics are not required.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import sys
from pathlib import Path

EXPECTED_PYTHON = (3, 14)
if sys.version_info[:2] != EXPECTED_PYTHON:
    raise RuntimeError(
        "This notebook requires CPython 3.14; "
        f"found {platform.python_version()}."
    )

import numpy as np

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "src" / "fastCGC" / "__init__.py").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the fastC-GC project root.")
SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from fastCGC.experiments.notebook_checkpoint import (
    configuration_hash,
    derive_algorithm_seed,
    file_sha256,
    reusable_success,
    write_checkpoint,
    write_npz_checkpoint,
)

In [ ]:
DATASET_KEY = "causalChamber"
METHOD_KEY = "cuts-plus"
MASTER_SEED = 42
ALPHA = 0.01
BETA = 0.001
N_PERMUTATIONS = 1000
BOOTSTRAP_SAMPLES = 200

# Pause/resume boundary: one complete method/dataset fit.
RESUME = True
MAX_NEW_FITS: int | None = None

DATASET_DIR = (
    PROJECT_ROOT / "data" / "shared" / "causal-comparison"
    / DATASET_KEY / "datasets"
)
OUTPUT_DIR = (
    PROJECT_ROOT / "data" / "outputs" / "causal-comparison"
    / DATASET_KEY / METHOD_KEY
)
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
NATIVE_OUTPUT_DIR = OUTPUT_DIR / "native"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_CONFIGURATION = {
    "dataset": DATASET_KEY,
    "method": METHOD_KEY,
    "master_seed": MASTER_SEED,
    "alpha": ALPHA,
    "beta": BETA,
    "n_permutations": N_PERMUTATIONS,
    "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "truth_scope": "physical reference network",
    **({"native_output_schema": 2} if METHOD_KEY.startswith("fast-") else {}),
}
CONFIG_HASH = configuration_hash(RUN_CONFIGURATION)
print(f"Dataset directory: {DATASET_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
from copy import deepcopy

from fastCGC.experiments.upstream import verify_upstream_revision

UNN_ROOT = Path(
    os.environ.get("UNN_ROOT", PROJECT_ROOT / "external" / "UNN")
).expanduser()
UPSTREAM_REVISION = verify_upstream_revision(
    UNN_ROOT,
    expected_url="https://github.com/jarrycyx/UNN.git",
    expected_commit="2212724cce8a7752db564089a7537fecedb0e36d",
)
CUTS_ROOT = UNN_ROOT / "CUTS_Plus"
if str(CUTS_ROOT) not in sys.path:
    sys.path.insert(0, str(CUTS_ROOT))

import torch
from omegaconf import OmegaConf
import cuts_plus
from utils.logger import MyLogger
from utils.misc import reproduc

SOURCE_REVISION = UPSTREAM_REVISION.commit


def run_native(series, fit_order, algorithm_seed, run_dir):
    options = OmegaConf.load(CUTS_ROOT / "opt" / "lorenz_example.yaml")
    native_options = deepcopy(options.sota.cuts_plus)
    native_options.n_nodes = int(series.shape[0])
    configured_groups = int(native_options.n_groups)
    native_options.n_groups = min(configured_groups, native_options.n_nodes)
    reproduc(seed=algorithm_seed, benchmark=False, deterministic=True)

    if torch.backends.mps.is_available():
        os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    log = MyLogger(
        log_dir=str(run_dir / "upstream-log"),
        stdout=False,
        stderr=False,
        tensorboard=False,
    )
    data = np.asarray(series.T, dtype=float)
    mask = np.ones_like(data)
    try:
        graph = cuts_plus.main(
            data,
            mask,
            None,
            native_options,
            log,
            device=device,
        )
    finally:
        log.close()
    return {"graph": np.asarray(graph)}

In [ ]:
bundle_paths = sorted(DATASET_DIR.glob("*.npz"))
if not bundle_paths:
    raise FileNotFoundError(
        f"No frozen {DATASET_KEY} bundles found in {DATASET_DIR}. "
        "Run 00-prepare-data.ipynb first."
    )

new_fits = 0
pause_requested = False
for bundle_path in bundle_paths:
    metadata_path = bundle_path.with_suffix(".json")
    if not metadata_path.is_file():
        raise FileNotFoundError(f"Missing metadata for {bundle_path}.")
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    with np.load(bundle_path, allow_pickle=False) as stored:
        series = np.asarray(stored["series"], dtype=float)
        if "lag_truth" in stored.files:
            default_order = int(stored["lag_truth"].shape[0])
        else:
            default_order = 1
    digest = hashlib.sha256(
        np.ascontiguousarray(series, dtype=np.float64).tobytes()
    ).hexdigest()
    recorded_digest = metadata.get("data_digest", metadata.get("digest"))
    if digest != recorded_digest:
        raise ValueError(f"Dataset digest mismatch for {bundle_path}.")
    fit_order = int(metadata.get("fit_order", default_order))

    checkpoint = CHECKPOINT_DIR / f"{bundle_path.stem}.json"
    expected = {
        "method": METHOD_KEY,
        "dataset": DATASET_KEY,
        "dataset_id": bundle_path.stem,
        "dataset_digest": digest,
        "fit_order": fit_order,
        "config_hash": CONFIG_HASH,
        "upstream_commit": SOURCE_REVISION,
    }
    if RESUME and reusable_success(checkpoint, expected=expected) is not None:
        print(f"{bundle_path.stem}: checkpoint reused")
        continue
    if MAX_NEW_FITS is not None and new_fits >= MAX_NEW_FITS:
        pause_requested = True
        break

    algorithm_seed = derive_algorithm_seed(
        "causal-comparison",
        MASTER_SEED,
        DATASET_KEY,
        METHOD_KEY,
        bundle_path.stem,
    )
    run_dir = OUTPUT_DIR / "runs" / bundle_path.stem
    run_dir.mkdir(parents=True, exist_ok=True)
    try:
        arrays = run_native(series, fit_order, algorithm_seed, run_dir)
        native_path = NATIVE_OUTPUT_DIR / f"{bundle_path.stem}.npz"
        native_digest = write_npz_checkpoint(native_path, arrays)
        write_checkpoint(
            checkpoint,
            {
                **expected,
                "status": "ok",
                "algorithm_seed": algorithm_seed,
                "native_output": str(native_path.relative_to(PROJECT_ROOT)),
                "native_output_sha256": native_digest,
                "error": "",
            },
        )
        new_fits += 1
        print(f"{bundle_path.stem}: complete")
    except KeyboardInterrupt:
        pause_requested = True
        print("Interrupted. Completed dataset checkpoints are safe.")
        break
    except Exception as error:
        write_checkpoint(
            checkpoint,
            {
                **expected,
                "status": "failed",
                "algorithm_seed": algorithm_seed,
                "native_output": "",
                "native_output_sha256": "",
                "error": f"{type(error).__name__}: {error}",
            },
        )
        print(f"{bundle_path.stem}: failed: {error}")

print(f"Completed {new_fits} new fits in this session.")
if pause_requested:
    print("Rerun this cell to resume at the next unfinished dataset.")